# Actividad 3.1 - Valores Atípicos
## Análisis de Piso (analisis_piso_sin_nulos.csv)

Detección y tratamiento de outliers con dos métodos: Desviación Estándar y Rango Intercuartílico (IQR).

### Paso 1: Importar librerías

In [ ]:
#Importamos las librerias pandas, numpy y matplotlib respectivamente
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Paso 2: Cargar el archivo CSV

In [ ]:
#Cargar archivo csv desde equipo
from google.colab import files
files.upload()

In [ ]:
#Carga desde un archivo .csv sin indice
data = pd.read_csv('analisis_piso_sin_nulos.csv')
data.head()

### Paso 3: Verificar estructura y valores nulos

In [ ]:
#Verificamos información del DataFrame
data.info()

In [ ]:
#Corroboramos valores nulos
valores_nulos = data.isnull().sum()
valores_nulos

### Paso 4: Limpiar Raking_Ventas (convertirla a numérica)

In [ ]:
#Reemplazamos "-" por NaN y convertimos la columna a numérica
data['Raking_Ventas'] = pd.to_numeric(data['Raking_Ventas'], errors='coerce')
data['Raking_Ventas'].dtype

### Paso 5: Separar columnas cuantitativas y cualitativas

In [ ]:
#Creo 2 dataframes para poder procesar los outliers
cuantitativas = data.iloc[:, 1:]
cualitativas = data.iloc[:, [0]]

### Paso 6: Diagrama de caja por cada columna

In [ ]:
#Diagrama de caja individual por cada columna (mejor que uno combinado, porque las escalas son muy distintas)
cols = cuantitativas.columns
n = len(cols)
n_cols_grid = 5
n_rows_grid = -(-n // n_cols_grid)  # redondeo hacia arriba

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(20, n_rows_grid * 2.5))
axes = axes.flatten()

for i, col in enumerate(cols):
    axes[i].boxplot(cuantitativas[col].dropna(), vert=False)
    axes[i].set_title(col, fontsize=8)
    axes[i].tick_params(labelsize=6)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## MÉTODO 1: DESVIACIÓN ESTÁNDAR

### Paso 7: Calcular límites (media ± 3 desviaciones estándar)

In [ ]:
#Método aplicando desviación estandar. Encuentro los valores extremos
y = cuantitativas
Limite_Superior = y.mean() + 3*y.std()
Limite_Inferior = y.mean() - 3*y.std()
print("Limite superior permitido", Limite_Superior)
print("Limite inferior permitido", Limite_Inferior)

### Paso 8: Marcar outliers como nulos

In [ ]:
#Obtenemos datos y los outliers se convierten en nulos en el DataFrame
data_std = cuantitativas[(y <= Limite_Superior) & (y >= Limite_Inferior)]
data_std

In [ ]:
#Corroboramos valores nulos del dataframe
valores_nulos = data_std.isnull().sum()
valores_nulos

### Paso 9: Rellenar outliers con la media

In [ ]:
#Reemplazamos valores atípicos (nulos) del dataframe con "mean"
#Realizamos una copia del dataframe
data_std_clean = data_std.copy()
data_std_clean = data_std_clean.fillna(round(data_std.mean(), 1))
data_std_clean

## MÉTODO 2: RANGO INTERCUARTÍLICO (IQR)

### Paso 10: Calcular límites (Q1 - 1.5·IQR, Q3 + 1.5·IQR)

In [ ]:
#Método aplicando Cuartiles. Encuentro cuartiles 0.25 y 0.75
percentile25 = y.quantile(0.25) #Q1
percentile75 = y.quantile(0.75) #Q3
iqr = percentile75 - percentile25

Limite_Superior_iqr = percentile75 + 1.5*iqr
Limite_Inferior_iqr = percentile25 - 1.5*iqr
print("Limite superior permitido", Limite_Superior_iqr)
print("Limite inferior permitido", Limite_Inferior_iqr)

### Paso 11: Identificar columnas donde el IQR es 0 (excluirlas del método)

In [ ]:
#Cuando IQR = 0, cualquier valor distinto de 0 se marcaría como outlier (no tiene sentido en columnas de ventas mensuales dispersas)
#Identificamos esas columnas para excluirlas del tratamiento
cols_iqr_valida = iqr[iqr > 0].index
cols_iqr_invalida = iqr[iqr == 0].index

print("Columnas EXCLUIDAS del método IQR (rango intercuartílico = 0):")
print(list(cols_iqr_invalida))

### Paso 12: Marcar outliers como nulos (solo columnas válidas)
**Nota:** `data_iqr` SÍ debe mostrar NaN aquí — son los outliers recién marcados. Es normal, se corrige en el paso siguiente.

In [ ]:
#Aplicamos el método IQR únicamente a las columnas donde el IQR > 0
data_iqr = cuantitativas.copy()
data_iqr[cols_iqr_valida] = cuantitativas[cols_iqr_valida][
    (y[cols_iqr_valida] <= Limite_Superior_iqr[cols_iqr_valida]) &
    (y[cols_iqr_valida] >= Limite_Inferior_iqr[cols_iqr_valida])
]
data_iqr

In [ ]:
#Corroboramos valores nulos del dataframe
valores_nulos = data_iqr.isnull().sum()
valores_nulos

### Paso 13: Rellenar outliers con la mediana
**Nota:** `data_iqr_clean` (después de esta celda) ya NO debe tener ningún NaN.

In [ ]:
#Reemplazamos valores atípicos (nulos) del dataframe con "median"
#Realizamos una copia del dataframe
data_iqr_clean = data_iqr.copy()
data_iqr_clean = data_iqr_clean.fillna(round(data_iqr.median(), 1))
data_iqr_clean

## RESULTADOS FINALES

### Paso 14: Unir cada dataframe limpio con la columna Asesor

In [ ]:
# Unimos el dataframe cuantitativo limpio con el dataframe cualitativo (Asesor)
Datos_limpios_std = pd.concat([cualitativas, data_std_clean], axis=1)
Datos_limpios_iqr = pd.concat([cualitativas, data_iqr_clean], axis=1)

Datos_limpios_std

### Paso 15: Exportar y descargar los CSVs

In [ ]:
# Convertimos ambos DataFrames a CSV
Datos_limpios_std.to_csv("analisis_piso_std.csv", index=False)
Datos_limpios_iqr.to_csv("analisis_piso_iqr.csv", index=False)

In [ ]:
# Descargamos ambos archivos
from google.colab import files
files.download("analisis_piso_std.csv")
files.download("analisis_piso_iqr.csv")